# 03 - Construccion OBT (One Big Table)

Objetivos:
- Consolidar un dataset analitico unico para BI y modelado.
- Agregar features temporales, geograficas y de negocio.
- Publicar la OBT en Snowflake para consumo en notebooks 04 y 05.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import datetime
import glob
import os

In [2]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = ['net.snowflake.spark.snowflake.DefaultSource', 'snowflake.DefaultSource']
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('03_construccion_obt')
assert_snowflake_connector(app)

curated_schema = os.environ.get('SF_CURATED_SCHEMA', 'CURATED')
analytics_schema = os.environ.get('SF_ANALYTICS_SCHEMA', 'ANALYTICS')

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

sf_utils = app._jvm.net.snowflake.spark.snowflake.Utils

def sf_run(sql):
    jsf = app._jvm.java.util.HashMap()
    for k, v in sf_option.items():
        jsf.put(k, v)
    sf_utils.runQuery(jsf, sql)

# El schema ANALYTICS debe existir antes de cualquier write Spark. No basta usar preactions,
# porque el conector puede validar el destino antes de ejecutar esas acciones.
sf_run(f'CREATE SCHEMA IF NOT EXISTS {analytics_schema}')

log_step(f'Notebook 03 ready: curated_schema={curated_schema}, analytics_schema={analytics_schema}')

[2026-04-07 22:38:31Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-07 22:38:37Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-07 22:38:43Z] Notebook 03 ready: curated_schema=CURATED, analytics_schema=ANALYTICS


In [3]:
# Base curada del notebook 02
def normalize_columns(df):
    return df.toDF(*[c.lower() for c in df.columns])

df_enriched = normalize_columns(
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', f'{curated_schema}.FCT_TRIPS_ENRICHED')
    .load()
)

curated_rows = df_enriched.count()
log_step(f'Loaded {curated_schema}.FCT_TRIPS_ENRICHED rows={curated_rows}')

[2026-04-07 22:39:21Z] Loaded CURATED.FCT_TRIPS_ENRICHED rows=446742


In [4]:
# Reglas de calidad minimas para la OBT
df_base = (
    df_enriched
    .filter(F.col('trip_nk').isNotNull())
    .filter(F.col('pickup_datetime_utc').isNotNull())
    .filter(F.col('dropoff_datetime_utc').isNotNull())
    .filter(F.col('total_amount').isNotNull())
    .filter(F.col('trip_duration_minutes').isNotNull())
    .filter(F.col('trip_duration_minutes') > 0)
)

base_rows = df_base.count()
log_step(f'Rows after OBT quality filters={base_rows}')

[2026-04-07 22:39:36Z] Rows after OBT quality filters=445964


In [5]:
# Features de tiempo, monetarias y de ruta
df_obt = (
    df_base
    .withColumn('pickup_datetime', F.col('pickup_datetime_utc'))
    .withColumn('dropoff_datetime', F.col('dropoff_datetime_utc'))
    .withColumn('pickup_date', F.to_date('pickup_datetime_utc'))
    .withColumn('dropoff_date', F.to_date('dropoff_datetime_utc'))
    .withColumn('trip_year', F.year('pickup_datetime_utc'))
    .withColumn('trip_month', F.month('pickup_datetime_utc'))
    .withColumn('year', F.year('pickup_datetime_utc'))
    .withColumn('month', F.month('pickup_datetime_utc'))
    .withColumn('trip_day', F.dayofmonth('pickup_datetime_utc'))
    .withColumn('trip_day_of_week', F.date_format('pickup_datetime_utc', 'E'))
    .withColumn('day_of_week', F.date_format('pickup_datetime_utc', 'E'))
    .withColumn('pickup_hour', F.hour('pickup_datetime_utc'))
    .withColumn('dropoff_hour', F.hour('dropoff_datetime_utc'))
    .withColumn(
        'pickup_time_band',
        F.when((F.col('pickup_hour') >= 6) & (F.col('pickup_hour') < 12), F.lit('morning'))
        .when((F.col('pickup_hour') >= 12) & (F.col('pickup_hour') < 18), F.lit('afternoon'))
        .when((F.col('pickup_hour') >= 18) & (F.col('pickup_hour') < 24), F.lit('evening'))
        .otherwise(F.lit('night'))
    )
    .withColumn('is_weekend', F.when(F.dayofweek('pickup_datetime_utc').isin([1, 7]), F.lit(1)).otherwise(F.lit(0)))
    .withColumn('source_service', F.col('service_type'))
    .withColumn('pu_location_id', F.col('pulocationid'))
    .withColumn('do_location_id', F.col('dolocationid'))
    .withColumn('vendor_name', F.col('vendor_desc'))
    .withColumn('payment_type', F.col('payment_type_code'))
    .withColumn('trip_type', F.col('trip_type_code'))
    .withColumn('store_and_fwd_flag', F.col('store_and_fwd_flag_norm'))
    .withColumn('trip_distance_miles', F.col('trip_distance').cast('double'))
    .withColumn('trip_distance_km', F.col('trip_distance').cast('double') * F.lit(1.60934))
    .withColumn(
        'distance_bucket',
        F.when(F.col('trip_distance_miles') < 1, F.lit('<1 mi'))
        .when((F.col('trip_distance_miles') >= 1) & (F.col('trip_distance_miles') < 3), F.lit('1-3 mi'))
        .when((F.col('trip_distance_miles') >= 3) & (F.col('trip_distance_miles') < 7), F.lit('3-7 mi'))
        .when((F.col('trip_distance_miles') >= 7) & (F.col('trip_distance_miles') < 15), F.lit('7-15 mi'))
        .otherwise(F.lit('15+ mi'))
    )
    .withColumn('avg_speed_mph', F.col('trip_distance_miles') / (F.col('trip_duration_minutes') / F.lit(60.0)))
    .withColumn('fare_per_mile', F.col('fare_amount') / F.when(F.col('trip_distance_miles') > 0, F.col('trip_distance_miles')).otherwise(F.lit(None)))
    .withColumn('tip_pct', (F.col('tip_amount') / F.when(F.col('fare_amount') > 0, F.col('fare_amount')).otherwise(F.lit(None))) * F.lit(100.0))
    .withColumn('route_key', F.concat_ws('->', F.coalesce(F.col('pu_zone'), F.lit('UNKNOWN')), F.coalesce(F.col('do_zone'), F.lit('UNKNOWN'))))
    .withColumn(
        'obt_trip_sk',
        F.sha2(
            F.concat_ws('||', F.col('trip_nk').cast('string'), F.col('trip_date').cast('string'), F.col('taxi_type').cast('string')),
            256
        )
    )
)

In [6]:
# Orden final de columnas para consumo analitico
obt_columns = [
    'obt_trip_sk', 'trip_nk',
    'taxi_type', 'service_type',
    'source_service',
    'pickup_datetime', 'dropoff_datetime', 'pickup_date', 'dropoff_date', 'dropoff_hour', 'day_of_week', 'month', 'year',
    'trip_date', 'trip_year', 'trip_month', 'trip_day', 'trip_day_of_week',
    'pickup_datetime_utc', 'dropoff_datetime_utc', 'pickup_hour', 'pickup_time_band', 'is_weekend',
    'pu_location_id', 'do_location_id',
    'pulocationid', 'pu_borough', 'pu_zone', 'pu_service_zone',
    'dolocationid', 'do_borough', 'do_zone', 'do_service_zone',
    'route_key',
    'vendor_id', 'vendor_desc', 'vendor_name',
    'rate_code_id', 'rate_code_desc',
    'payment_type_code', 'payment_type', 'payment_type_desc',
    'trip_type_code', 'trip_type', 'trip_type_desc',
    'store_and_fwd_flag', 'store_and_fwd_flag_norm', 'store_and_fwd_desc',
    'passenger_count',
    'trip_distance', 'trip_distance_miles', 'trip_distance_km', 'distance_bucket', 'trip_duration_minutes', 'avg_speed_mph',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount',
    'fare_per_mile', 'tip_pct',
    'source_year', 'source_month', 'run_id', 'ingested_at_utc'
]

typed_defaults = {
    'obt_trip_sk': 'string', 'trip_nk': 'string', 'taxi_type': 'string', 'service_type': 'string', 'source_service': 'string',
    'pickup_datetime': 'timestamp', 'dropoff_datetime': 'timestamp', 'pickup_date': 'date', 'dropoff_date': 'date', 'dropoff_hour': 'int',
    'day_of_week': 'string', 'month': 'int', 'year': 'int', 'trip_date': 'date', 'trip_year': 'int', 'trip_month': 'int',
    'trip_day': 'int', 'trip_day_of_week': 'string', 'pickup_datetime_utc': 'timestamp', 'dropoff_datetime_utc': 'timestamp',
    'pickup_hour': 'int', 'pickup_time_band': 'string', 'is_weekend': 'int', 'pu_location_id': 'int', 'do_location_id': 'int',
    'pulocationid': 'int', 'pu_borough': 'string', 'pu_zone': 'string', 'pu_service_zone': 'string',
    'dolocationid': 'int', 'do_borough': 'string', 'do_zone': 'string', 'do_service_zone': 'string', 'route_key': 'string',
    'vendor_id': 'int', 'vendor_desc': 'string', 'vendor_name': 'string', 'rate_code_id': 'int', 'rate_code_desc': 'string',
    'payment_type_code': 'int', 'payment_type': 'int', 'payment_type_desc': 'string', 'trip_type_code': 'int', 'trip_type': 'int',
    'trip_type_desc': 'string', 'store_and_fwd_flag': 'string', 'store_and_fwd_flag_norm': 'string', 'store_and_fwd_desc': 'string',
    'passenger_count': 'double', 'trip_distance': 'double', 'trip_distance_miles': 'double', 'trip_distance_km': 'double',
    'distance_bucket': 'string', 'trip_duration_minutes': 'double', 'avg_speed_mph': 'double', 'fare_amount': 'double',
    'extra': 'double', 'mta_tax': 'double', 'tip_amount': 'double', 'tolls_amount': 'double', 'improvement_surcharge': 'double',
    'congestion_surcharge': 'double', 'airport_fee': 'double', 'total_amount': 'double', 'fare_per_mile': 'double', 'tip_pct': 'double',
    'source_year': 'int', 'source_month': 'int', 'run_id': 'string', 'ingested_at_utc': 'timestamp'
}

for c in obt_columns:
    if c not in df_obt.columns:
        df_obt = df_obt.withColumn(c, F.lit(None).cast(typed_defaults.get(c, 'string')))

df_obt_final = df_obt.select(*obt_columns)
print('Rows OBT final:', df_obt_final.count())

Rows OBT final: 445964


In [7]:
# Publicacion de la OBT principal
sf_run(f'CREATE SCHEMA IF NOT EXISTS {analytics_schema}')

(
    df_obt_final
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', f'{analytics_schema}.OBT_TRIPS')
    .mode('overwrite')
    .save()
)

In [8]:
# Agregado mensual de apoyo para dashboards
sf_run(f'CREATE SCHEMA IF NOT EXISTS {analytics_schema}')

df_obt_monthly = (
    df_obt_final
    .groupBy('trip_year', 'trip_month', 'taxi_type', 'pu_borough', 'do_borough', 'payment_type_desc')
    .agg(
        F.count('*').alias('trips_count'),
        F.sum('total_amount').alias('total_revenue'),
        F.avg('total_amount').alias('avg_ticket'),
        F.avg('trip_distance_miles').alias('avg_distance_miles'),
        F.avg('trip_duration_minutes').alias('avg_duration_minutes'),
        F.avg('tip_pct').alias('avg_tip_pct')
    )
)

(
    df_obt_monthly
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', f'{analytics_schema}.OBT_TRIPS_MONTHLY')
    .mode('overwrite')
    .save()
)

In [9]:
# Validaciones rapidas
df_obt_final.groupBy('taxi_type').count().orderBy('taxi_type').show()

df_obt_final.select(
    F.count('*').alias('rows_obt'),
    F.approx_count_distinct('trip_nk').alias('approx_distinct_trip_nk'),
    F.sum(F.when(F.col('pu_zone').isNull(), 1).otherwise(0)).alias('null_pu_zone'),
    F.sum(F.when(F.col('do_zone').isNull(), 1).otherwise(0)).alias('null_do_zone')
).show()

df_obt_final.select(
    'trip_nk', 'taxi_type', 'trip_date', 'pickup_time_band',
    'pu_borough', 'pu_zone', 'do_borough', 'do_zone',
    'distance_bucket', 'trip_distance_miles', 'trip_duration_minutes', 'avg_speed_mph',
    'payment_type_desc', 'total_amount', 'tip_pct', 'route_key'
).show(20, truncate=False)

+---------+------+
|taxi_type| count|
+---------+------+
|    green|445964|
+---------+------+

+--------+-----------------------+------------+------------+
|rows_obt|approx_distinct_trip_nk|null_pu_zone|null_do_zone|
+--------+-----------------------+------------+------------+
|  445964|                 444375|           0|           0|
+--------+-----------------------+------------+------------+

+----------------------------------------------------------------+---------+----------+----------------+----------+--------------------------------+----------+---------------------------------+---------------+-------------------+---------------------+------------------+-----------------+------------+------------------+--------------------------------------------------------------+
|trip_nk                                                         |taxi_type|trip_date |pickup_time_band|pu_borough|pu_zone                         |do_borough|do_zone                          |distance_bucket|trip_

In [10]:
# Verificacion explicita de idempotencia sobre un subconjunto deterministico.
# La prueba se ejecuta siempre; no depende de parametros externos.
idempotency_subset = (
    df_obt_final
    .select('source_year', 'source_month', 'service_type')
    .where(
        F.col('source_year').isNotNull()
        & F.col('source_month').isNotNull()
        & F.col('service_type').isNotNull()
    )
    .groupBy('source_year', 'source_month', 'service_type')
    .count()
    .orderBy('source_year', 'source_month', 'service_type')
    .first()
)

if idempotency_subset is None:
    raise ValueError('No hay datos en df_obt_final para seleccionar un subconjunto de idempotencia.')

idempotency_test_year = int(idempotency_subset['source_year'])
idempotency_test_month = int(idempotency_subset['source_month'])
idempotency_test_service_type = str(idempotency_subset['service_type']).lower()
log_step(
    'Idempotency subset: '
    f'year={idempotency_test_year}, month={idempotency_test_month}, '
    f'service_type={idempotency_test_service_type}'
)


def filter_idempotency_subset(df):
    return df.filter(
        (F.col('source_year').cast('int') == F.lit(idempotency_test_year))
        & (F.col('source_month').cast('int') == F.lit(idempotency_test_month))
        & (F.lower(F.col('service_type')) == F.lit(idempotency_test_service_type))
    )


def idempotency_metrics(df, metric_set):
    return (
        filter_idempotency_subset(df)
        .agg(
            F.count(F.lit(1)).alias('row_count'),
            F.countDistinct('trip_nk').alias('distinct_trip_nk'),
            F.countDistinct('obt_trip_sk').alias('distinct_obt_trip_sk'),
        )
        .withColumn('metric_set', F.lit(metric_set))
        .withColumn('duplicate_trip_nk_rows', F.col('row_count') - F.col('distinct_trip_nk'))
        .withColumn('duplicate_obt_trip_sk_rows', F.col('row_count') - F.col('distinct_obt_trip_sk'))
        .select(
            'metric_set',
            'row_count',
            'distinct_trip_nk',
            'distinct_obt_trip_sk',
            'duplicate_trip_nk_rows',
            'duplicate_obt_trip_sk_rows',
        )
    )


# Antes: subconjunto ya publicado por el full refresh overwrite en analytics.OBT_TRIPS.
df_obt_published = normalize_columns(
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', f'{analytics_schema}.OBT_TRIPS')
    .load()
)

# Despues: el mismo subconjunto reconstruido por la logica actual del notebook en df_obt_final.
df_idempotency_before = idempotency_metrics(df_obt_published, 'before_published_obt')
df_idempotency_after = idempotency_metrics(df_obt_final, 'after_rebuilt_same_logic')
df_idempotency_metrics = df_idempotency_before.unionByName(df_idempotency_after)

df_idempotency_metrics.show(truncate=False)

metrics_by_set = {row['metric_set']: row.asDict() for row in df_idempotency_metrics.collect()}
before_metrics = metrics_by_set['before_published_obt']
after_metrics = metrics_by_set['after_rebuilt_same_logic']

published_keys = filter_idempotency_subset(df_obt_published).select('trip_nk', 'obt_trip_sk').dropDuplicates()
rebuilt_keys = filter_idempotency_subset(df_obt_final).select('trip_nk', 'obt_trip_sk').dropDuplicates()
missing_after_count = published_keys.exceptAll(rebuilt_keys).count()
new_after_count = rebuilt_keys.exceptAll(published_keys).count()

idempotency_comparison = app.createDataFrame([
    (
        idempotency_test_year,
        idempotency_test_month,
        idempotency_test_service_type,
        before_metrics['row_count'],
        after_metrics['row_count'],
        after_metrics['row_count'] - before_metrics['row_count'],
        before_metrics['distinct_trip_nk'],
        after_metrics['distinct_trip_nk'],
        after_metrics['distinct_trip_nk'] - before_metrics['distinct_trip_nk'],
        before_metrics['distinct_obt_trip_sk'],
        after_metrics['distinct_obt_trip_sk'],
        after_metrics['distinct_obt_trip_sk'] - before_metrics['distinct_obt_trip_sk'],
        missing_after_count,
        new_after_count,
    )
], [
    'test_year',
    'test_month',
    'test_service_type',
    'before_row_count',
    'after_row_count',
    'row_count_delta',
    'before_distinct_trip_nk',
    'after_distinct_trip_nk',
    'distinct_trip_nk_delta',
    'before_distinct_obt_trip_sk',
    'after_distinct_obt_trip_sk',
    'distinct_obt_trip_sk_delta',
    'published_keys_missing_after_rebuild',
    'rebuilt_keys_not_in_published',
])

idempotency_comparison.show(truncate=False)

if before_metrics['row_count'] == 0:
    raise AssertionError('El subconjunto deterministico de idempotencia no tiene filas.')

for metric_name in ['row_count', 'distinct_trip_nk', 'distinct_obt_trip_sk']:
    if before_metrics[metric_name] != after_metrics[metric_name]:
        raise AssertionError(f'Idempotencia fallida: {metric_name} cambio entre antes y despues.')

if before_metrics['duplicate_trip_nk_rows'] != 0 or after_metrics['duplicate_trip_nk_rows'] != 0:
    raise AssertionError('Idempotencia fallida: existen duplicados por trip_nk en el subconjunto evaluado.')

if before_metrics['duplicate_obt_trip_sk_rows'] != 0 or after_metrics['duplicate_obt_trip_sk_rows'] != 0:
    raise AssertionError('Idempotencia fallida: existen duplicados por obt_trip_sk en el subconjunto evaluado.')

if missing_after_count != 0 or new_after_count != 0:
    raise AssertionError('Idempotencia fallida: el set de claves cambia al reconstruir el mismo subconjunto.')

log_step('Idempotency verification passed for analytics.OBT_TRIPS subset.')


[2026-04-07 22:41:23Z] Idempotency subset: year=2020, month=1, service_type=green
+------------------------+---------+----------------+--------------------+----------------------+--------------------------+
|metric_set              |row_count|distinct_trip_nk|distinct_obt_trip_sk|duplicate_trip_nk_rows|duplicate_obt_trip_sk_rows|
+------------------------+---------+----------------+--------------------+----------------------+--------------------------+
|before_published_obt    |445964   |445964          |445964              |0                     |0                         |
|after_rebuilt_same_logic|445964   |445964          |445964              |0                     |0                         |
+------------------------+---------+----------------+--------------------+----------------------+--------------------------+

+---------+----------+-----------------+----------------+---------------+---------------+-----------------------+----------------------+----------------------+--------

# Verificación de idempotencia

Se probo un subconjunto deterministico de `analytics.OBT_TRIPS`, seleccionado automaticamente como el primer `source_year`, `source_month` y `service_type` disponible en la OBT.

La prueba se ejecuta siempre y compara el subconjunto ya publicado en Snowflake despues del `mode('overwrite')` contra el mismo subconjunto reconstruido por la logica actual del notebook en `df_obt_final`. Para ambos lados se muestran `row_count`, `count distinct trip_nk` y `count distinct obt_trip_sk`.

El resultado esperado es que los conteos no cambien, que no existan duplicados por `trip_nk` ni por `obt_trip_sk`, y que el set de claves publicado sea igual al set reconstruido.

Esto demuestra idempotencia de la construccion porque la OBT se publica como full refresh con `overwrite`: al reejecutar el notebook con el mismo input, Snowflake reemplaza la tabla en lugar de anexar filas, y el subconjunto evaluado conserva la misma cardinalidad y las mismas claves.
